# Margin-gated selective reranking

**The method.** Reranking every query is effective but costs a full cross-encoder pass per candidate, on every request. But this project's diagnostic work produced a signal that predicts when retrieval is about to fail — and it's **free**, computed from scores the retriever already has:

```
margin(q) = s(top1) - s(top2)
```

```
retrieve  →  margin (free)
              ├─ margin >  τ  → confident, skip the reranker
              └─ margin ≤ τ  → ambiguous, invoke the reranker
```

**Why it's dialect-aware without dialect detection:** Darija queries sit systematically closer to the decision boundary (measured earlier: +0.274 margin for MSA vs +0.040 for Darija), so they get routed to the reranker more often — automatically. No language ID, no extra model, no training.

**Note on the margin:** the diagnostics used `s(gold) − s(best_other)`, which needs the label and isn't deployable. This uses `s(top1) − s(top2)`, which needs no label and is computable at query time.

**Cell 6 checks the premise first** — if Darija margins aren't actually lower, the method's central claim fails and that gets reported instead.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **GPU required.**

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "reranker": "BAAI/bge-reranker-v2-m3",
    "alpha": 0.8,
    "retrieve_k": 20,
    "k_values": (1, 3, 5),
    "bootstrap_n": 1000,
    "seed": 42,
}

### Load data

In [ ]:
import json, random, re, gc, time
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
print(f"Corpus {len(corpus)} | evaluating all {len(qa)} items")

### BM25 + hybrid scoring

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Stage 1: retrieve, and record the DEPLOYABLE margin

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

print("Building index...")
bi = SentenceTransformer(CONFIG["base_encoder"])
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")

def retrieve_with_margin(query, k):
    """Returns (candidate_ids, margin). The margin is s(top1) - s(top2):
    no gold label involved, so it is computable at inference time."""
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = (CONFIG["alpha"] * minmax(corpus_emb @ q)
         + (1 - CONFIG["alpha"]) * minmax(np.asarray(bm25.get_scores(tokenize(query)))))
    order = np.argsort(-s)[:k]
    margin = float(s[order[0]] - s[order[1]]) if len(order) > 1 else 1.0
    return [corpus_ids[i] for i in order], margin

K = CONFIG["retrieve_k"]
retrieved = {}
for field in ["msa_query", "darija_query"]:
    retrieved[field] = {}
    for q in qa:
        cands, margin = retrieve_with_margin(q[field], K)
        retrieved[field][q["id"]] = {"candidates": cands, "margin": margin}
    margins = [v["margin"] for v in retrieved[field].values()]
    print(f"  {field:<14} mean top1-top2 margin = {np.mean(margins):.4f}  "
          f"median = {np.median(margins):.4f}")

del bi, corpus_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Is the margin systematically lower for Darija?

In [ ]:
# This is the premise the whole method rests on: if dialectal queries did not
# have lower margins, gating on the margin would not be dialect-aware.
rng = np.random.default_rng(CONFIG["seed"])
m_msa = np.array([retrieved["msa_query"][q["id"]]["margin"] for q in qa])
m_dar = np.array([retrieved["darija_query"][q["id"]]["margin"] for q in qa])

d = m_msa - m_dar
idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
lo, hi = np.percentile(d[idx].mean(axis=1), [2.5, 97.5])

print("=" * 78)
print("PREMISE CHECK — do dialectal queries have lower retrieval margins?")
print("=" * 78)
print(f"  MSA    mean margin  {m_msa.mean():.4f}")
print(f"  Darija mean margin  {m_dar.mean():.4f}")
print(f"  difference          {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  "
      f"{'CONFIRMED' if lo > 0 else 'NOT CONFIRMED'}")
if lo <= 0:
    print("\n  If not confirmed, margin-gating is not dialect-aware and the method's")
    print("  main claim does not hold -- report this rather than the intended result.")

### Stage 2: rerank EVERYTHING once, so any threshold can be

In [ ]:
# evaluated afterwards without re-running the cross-encoder
from sentence_transformers import CrossEncoder

print("\nReranking all queries once (results reused for every threshold)...")
ce = CrossEncoder(CONFIG["reranker"], max_length=512, trust_remote_code=True,
                  automodel_args={"torch_dtype": torch.float32})

reranked = {}
for field in ["msa_query", "darija_query"]:
    reranked[field] = {}
    for q in qa:
        cands = retrieved[field][q["id"]]["candidates"]
        pairs = [(q[field], corpus_map[c]) for c in cands]
        scores = np.asarray(ce.predict(pairs, batch_size=16, show_progress_bar=False))
        if scores.ndim > 1:          # some rerankers return per-class scores
            scores = scores[:, -1]
        order = np.argsort(-scores)
        reranked[field][q["id"]] = [cands[i] for i in order]
    print(f"  {field} reranked")

del ce
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Evaluate any gating threshold

In [ ]:
def evaluate_gated(field, tau):
    """Rerank only queries whose margin <= tau; keep retrieval order otherwise."""
    hits = {k: [] for k in CONFIG["k_values"]}
    rr, reranked_flags = [], []
    for q in qa:
        info = retrieved[field][q["id"]]
        use_rerank = info["margin"] <= tau
        ordered = reranked[field][q["id"]] if use_rerank else info["candidates"]
        reranked_flags.append(int(use_rerank))
        gold = q["source_chunk_id"]
        pos = ordered.index(gold) + 1 if gold in ordered else None
        for k in CONFIG["k_values"]:
            hits[k].append(1.0 if (pos and pos <= k) else 0.0)
        rr.append(1.0 / pos if pos else 0.0)
    return {
        **{f"R@{k}": np.array(v) for k, v in hits.items()},
        "MRR": np.array(rr),
        "reranked": np.array(reranked_flags),
    }

# tau = 0 -> never rerank (retrieval only);  tau = inf -> always rerank
TAUS = [0.0, 0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.30, 0.50, np.inf]

rows = []
for field in ["msa_query", "darija_query"]:
    for tau in TAUS:
        r = evaluate_gated(field, tau)
        rows.append({
            "query": field, "tau": tau,
            "R@1": r["R@1"].mean(), "R@5": r["R@5"].mean(), "MRR": r["MRR"].mean(),
            "frac_reranked": r["reranked"].mean(),
        })
curve = pd.DataFrame(rows)
curve.to_csv("margin_gating_curve.csv", index=False)

print("=" * 78)
print("GATING CURVE")
print("=" * 78)
for field in ["msa_query", "darija_query"]:
    print(f"\n--- {field} ---")
    sub = curve[curve["query"] == field]
    print(sub[["tau", "R@1", "R@5", "MRR", "frac_reranked"]]
          .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

### The dialect-aware claim: is Darija reranked more often?

In [ ]:
print("\n" + "=" * 78)
print("IS GATING DIALECT-AWARE? (fraction of queries sent to the reranker)")
print("=" * 78)
pivot = curve.pivot(index="tau", columns="query", values="frac_reranked")
pivot["darija_minus_msa"] = pivot["darija_query"] - pivot["msa_query"]
print(pivot.to_string(float_format=lambda x: f"{x:.3f}"))
print("""
A positive 'darija_minus_msa' means dialectal queries are routed to the reranker
more often than MSA queries at the same threshold -- the gate adapts to dialect
without being told which queries are dialectal.""")

### Operating points: how much benefit is kept, at what cost

In [ ]:
print("\n" + "=" * 78)
print("OPERATING POINTS (Darija)")
print("=" * 78)
dar = curve[curve["query"] == "darija_query"].set_index("tau")
never, always = dar.loc[0.0], dar.loc[np.inf]
print(f"  never rerank   R@1={never['R@1']:.3f}  (0% reranked)")
print(f"  always rerank  R@1={always['R@1']:.3f}  (100% reranked)")
full_gain = always["R@1"] - never["R@1"]
print(f"  full gain from always reranking: {full_gain:+.3f}\n")

op = []
for tau in TAUS:
    if tau in (0.0, np.inf):
        continue
    row = dar.loc[tau]
    kept = (row["R@1"] - never["R@1"]) / full_gain * 100 if full_gain > 0 else float("nan")
    op.append({"tau": tau, "R@1": row["R@1"], "frac_reranked": row["frac_reranked"],
               "benefit_kept_%": kept,
               "compute_saved_%": (1 - row["frac_reranked"]) * 100})
opdf = pd.DataFrame(op)
print(opdf.to_string(index=False, float_format=lambda x: f"{x:.1f}"))
opdf.to_csv("margin_gating_operating_points.csv", index=False)

# The headline operating point: most of the benefit for the least compute.
good = opdf[opdf["benefit_kept_%"] >= 90]
if len(good):
    best = good.loc[good["frac_reranked"].idxmin()]
    print(f"""
HEADLINE: at tau = {best['tau']:.2f}, {best['benefit_kept_%']:.0f}% of the reranking
benefit is retained while reranking only {best['frac_reranked']*100:.0f}% of queries
({best['compute_saved_%']:.0f}% of reranking compute saved).""")
else:
    print("\nNo threshold retained >=90% of the benefit; report the full curve instead.")

### Effect on the dialect gap across thresholds

In [ ]:
print("\n" + "=" * 78)
print("DIALECT GAP ACROSS THRESHOLDS")
print("=" * 78)
gap = curve.pivot(index="tau", columns="query", values="R@1")
gap["gap"] = gap["msa_query"] - gap["darija_query"]
gap["mean_frac_reranked"] = curve.groupby("tau")["frac_reranked"].mean()
print(gap.to_string(float_format=lambda x: f"{x:.3f}"))
gap.to_csv("margin_gating_gap.csv")

from google.colab import files
files.download("margin_gating_curve.csv")
files.download("margin_gating_operating_points.csv")
files.download("margin_gating_gap.csv")